# 🧠 Nested Comprehensions & Data Structure Flattening: Deep-Dive Guide

Welcome to the **Nested Comprehensions & Data Structure Flattening Mastery Notebook**.

Nested comprehensions are among the most powerful features in Python for data scientists and fintech engineers. They allow you to transform and flatten complex, multi-layered data structures (like YAML configs, nested JSON responses, and API trees) into clean lists, sets, and tuples in a single, high-speed line of code.

---

### 💡 The Golden Rule for Beginners:
> **A Python Comprehension is just a traditional nested loop written in one line from left to right:**

```text
┌─ Visual Reading Direction ────────────────────────────────────────────────────────┐
│                                                                                    │
│   { target_item   FOR outer_item IN outer_list   FOR target_item IN inner_list }   │
│     ▲             └──────────┬───────────────┘   └──────────┬────────────────┘     │
│     │                        │                              │                      │
│  1. What you want      2. Outer Loop                  3. Inner Loop                │
│     to extract            (Entity Level)                 (Item Level)              │
└────────────────────────────────────────────────────────────────────────────────────┘
```

---

### 📚 Key Topics Covered in this Notebook:
- [x] 🔹 **The Mental Model:** Converting 6-line traditional nested loops into 1-line comprehensions
- [x] 🔹 **Set Comprehensions (`{...}`):** Automatic deduplication of nested lists
- [x] 🔹 **Tuple / Generator Ingestion (`tuple(...)`):** Creating immutable prefix collections
- [x] 🔹 **Dictionary Comprehensions (`{k: v for ...}`):** Inverting and mapping nested keys
- [x] 🔹 **Real Fintech Example:** Flattening YAML OFAC Sanctions & Nested JSON Telemetry

In [ ]:
# Setup: Sample Nested Fintech Watchlist Configuration
sanctions_cfg = [
    {
        'entity_id': 'SANC-101',
        'name': 'Volkov Cyber Syndicate',
        'high_risk_countries': ['RU', 'BY', 'SG'],
        'linked_ip_prefixes': ['213.238.', '203.0.113.']
    },
    {
        'entity_id': 'SANC-102',
        'name': 'Alpha Global Remittance Corp',
        'high_risk_countries': ['MX', 'CO', 'PA', 'FR'],
        'linked_ip_prefixes': ['39.175.', '192.0.2.']
    },
    {
        'entity_id': 'SANC-103',
        'name': 'PetroCasas International',
        'high_risk_countries': ['VE', 'CU', 'DE'],
        'linked_ip_prefixes': ['29.178.', '198.18.0.']
    }
]

print(f"Loaded {len(sanctions_cfg)} nested entity records.")

---
## 1. Traditional Nested `for` Loop vs. 1-Line Set Comprehension

### Goal: Extract all unique country codes from every entity into a clean `set`.

### 🔹 Approach A: The Traditional 6-Line Loop

In [ ]:
# Approach A: Traditional Nested For-Loop with .add()
sanctioned_countries_loop = set()

for entity in sanctions_cfg:                                   # Outer loop
    country_list = entity.get('high_risk_countries', [])      # Get inner list
    for country in country_list:                              # Inner loop
        sanctioned_countries_loop.add(country)                # Collect item

print("Traditional Loop Result:", sanctioned_countries_loop)

### 🔹 Approach B: The 1-Line Set Comprehension
Notice how the order of `for` statements in the comprehension is **identical** to the indentation order of the traditional loop:

In [ ]:
# Approach B: Set Comprehension (1-Liner)
sanctioned_countries_comp = {country for entity in sanctions_cfg for country in entity.get('high_risk_countries', [])}

print("Set Comprehension Result:", sanctioned_countries_comp)

# Prove both approaches yield the exact same output
assert sanctioned_countries_loop == sanctioned_countries_comp, "Outputs do not match!"
print("✅ Both methods are 100% equivalent!")

---
## 2. Extracting Nested Strings into a `tuple` (for String Matching)

When using string functions like Pandas `.str.startswith()`, we need a **tuple** of prefixes.

### 🔹 Approach A: The Traditional Loop with `.append()`

In [ ]:
# Traditional Loop
ip_list = []
for entity in sanctions_cfg:
    for ip in entity.get('linked_ip_prefixes', []):
        ip_list.append(ip)

sanctioned_ips_loop = tuple(ip_list)
print("Traditional Tuple of IPs:", sanctioned_ips_loop)

### 🔹 Approach B: The 1-Line Tuple Generator Expression

In [ ]:
# 1-Line Tuple Comprehension
sanctioned_ips_comp = tuple(ip for entity in sanctions_cfg for ip in entity.get('linked_ip_prefixes', []))
print("1-Liner Tuple of IPs:", sanctioned_ips_comp)

assert sanctioned_ips_loop == sanctioned_ips_comp
print("✅ Both methods are 100% equivalent!")

---
## 3. Nested Dictionary Comprehensions (Inverted Lookups)

### Scenario: Create a fast lookup map where `country_code -> entity_name`

In [ ]:
# Nested Dict Comprehension: { country: entity_name }
country_to_entity = {
    country: entity['name']
    for entity in sanctions_cfg
    for country in entity.get('high_risk_countries', [])
}

print("Country to Sanction Entity Lookup Map:")
for country, name in country_to_entity.items():
    print(f"  • {country} -> {name}")

assert country_to_entity['RU'] == 'Volkov Cyber Syndicate'
assert country_to_entity['FR'] == 'Alpha Global Remittance Corp'
assert country_to_entity['DE'] == 'PetroCasas International'
print("✅ Dict Comprehension Verified!")

---
## 4. Cheat Sheet: Comprehension Syntax Summary

| Comprehension Type | Output Data Type | Syntax Pattern | Best Used For |
| :--- | :--- | :--- | :--- |
| **List Comprehension** | `list` | `[item for outer in obj for item in outer.sublist]` | Preserving duplicate rows & order |
| **Set Comprehension** | `set` | `{item for outer in obj for item in outer.sublist}` | **Automatic deduplication** (IDs, countries) |
| **Dict Comprehension** | `dict` | `{item: outer.val for outer in obj for item in outer.sublist}` | Inverted O(1) key-value lookup tables |
| **Tuple Generator** | `tuple` | `tuple(item for outer in obj for item in outer.sublist)` | String prefix matches (`.startswith()`) |